# Real-Time Safety Monitor with Alert System


## Setup

In [ ]:
import torch
import cv2
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import numpy as np
from datetime import datetime
import time
import os
import json
import threading
import subprocess


os.makedirs('alerts', exist_ok=True)
os.makedirs('logs', exist_ok=True)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Alert system: Mac notifications + system sounds")

## Load Model

In [ ]:
model_name = "Salesforce/blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(model_name)
model = BlipForConditionalGeneration.from_pretrained(model_name).to(device)
model.eval()

print("BLIP model loaded")

## Alert System

In [ ]:
class AlertSystem:
    """Manages alerts with visible popup dialogs"""
    
    def __init__(self):
        self.alert_log = []
        self.last_alert_time = 0
        self.alert_cooldown = 5.0
        self.total_alerts = 0
        
    def should_alert(self):
        """Check if enough time has passed since last alert"""
        current_time = time.time()
        if current_time - self.last_alert_time >= self.alert_cooldown:
            self.last_alert_time = current_time
            return True
        return False
    
    def trigger_alert(self, caption, hazard_type, severity, frame):
        """Trigger all alert mechanisms"""
        if not self.should_alert():
            return
        
        timestamp = datetime.now()
        self.total_alerts += 1
        
        # Log alert
        alert_data = {
            'timestamp': timestamp.strftime('%Y-%m-%d %H:%M:%S'),
            'caption': caption,
            'hazard_type': hazard_type,
            'severity': severity
        }
        self.alert_log.append(alert_data)
        
        # Save screenshot
        screenshot_path = f"alerts/hazard_{timestamp.strftime('%Y%m%d_%H%M%S')}.jpg"
        cv2.imwrite(screenshot_path, frame)
        
        # Visual popup dialog
        threading.Thread(target=self._show_popup, args=(hazard_type, caption, severity), daemon=True).start()
        
        # Sound alert
        threading.Thread(target=self._play_alert_sound, daemon=True).start()
        
        # Console alert 
        print("\n" + "=" * 40)
        print(f"   ALERT #{self.total_alerts} - {hazard_type.upper()}")
        print(f"   Severity: {severity.upper()}")
        print(f"   {caption}")
        print(f"   Screenshot: {screenshot_path}")
        print("=" * 40 + "\n")
    
    def _show_popup(self, title, message, severity):
        """Show popup dialog in center of screen"""
        try:
            # Truncate message
            message_short = message[:200]
            
            # Icon based on severity
            icon = "stop" if severity == "high" else "caution"
            
            # Create popup that auto-closes after 3 seconds
            script = f'''
            display dialog "⚠️  HAZARD DETECTED

Type: {title}

{message_short}" ¬
            buttons {{"OK"}} ¬
            default button "OK" ¬
            with icon {icon} ¬
            giving up after 3
            '''
            
            subprocess.run(['osascript', '-e', script], 
                         check=False, 
                         capture_output=True, 
                         timeout=5)
        except Exception as e:
            pass
    
    def _play_alert_sound(self):
        """Play system beep"""
        try:
            os.system('afplay /System/Library/Sounds/Basso.aiff &')
        except:
            print('\a')
    
    def save_log(self):
        """Save alert log to file"""
        if self.alert_log:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            log_file = f"logs/alert_log_{timestamp}.json"
            with open(log_file, 'w') as f:
                json.dump(self.alert_log, f, indent=2)
            return log_file
        return None
    
    def get_stats(self):
        """Return alert statistics"""
        return {
            'total_alerts': self.total_alerts,
            'alert_count': len(self.alert_log),
            'hazard_types': self._count_hazard_types()
        }
    
    def _count_hazard_types(self):
        """Count alerts by hazard type"""
        counts = {}
        for alert in self.alert_log:
            hazard = alert['hazard_type']
            counts[hazard] = counts.get(hazard, 0) + 1
        return counts

alert_system = AlertSystem()
print("Alert system initialized (with popup dialogs)")

## Hazard Classifier

In [ ]:
class SafetyClassifier:
    def __init__(self):
        self.hazard_keywords = {
            'electrical': ['wire', 'cable', 'cord', 'plug', 'socket', 'outlet', 'electrical', 'power'],
            'trip': ['floor', 'ground', 'lying', 'scattered', 'pile', 'clutter', 'mess'],
            'slip': ['wet', 'water', 'liquid', 'spill', 'slippery', 'puddle'],
            'fire': ['fire', 'flame', 'smoke', 'burning', 'hot'],
            'sharp': ['glass', 'broken', 'sharp', 'shattered', 'crack'],
            'blocked': ['blocked', 'obstruct', 'narrow', 'crowded']
        }
        
        self.safe_keywords = ['clean', 'organized', 'tidy', 'clear', 'empty', 'neat', 'spacious']
    
    def classify(self, caption):
        caption_lower = caption.lower()
        
        # Check for safe indicators
        has_safe = any(keyword in caption_lower for keyword in self.safe_keywords)
        
        # Check for hazards and identify type
        detected_hazards = []
        for hazard_type, keywords in self.hazard_keywords.items():
            if any(keyword in caption_lower for keyword in keywords):
                detected_hazards.append(hazard_type)
        
        if detected_hazards:
            hazard_str = ', '.join(detected_hazards)
            # Determine severity
            if any(h in ['electrical', 'fire', 'sharp'] for h in detected_hazards):
                return 'hazard', hazard_str, 'high', (0, 0, 255)
            else:
                return 'hazard', hazard_str, 'medium', (0, 165, 255)
        
        if has_safe:
            return 'safe', 'none', 'none', (0, 255, 0)
        
        # Default: assume hazard for safety
        return 'hazard', 'unknown', 'low', (255, 255, 0)

classifier = SafetyClassifier()
print("Classifier ready")

## Caption Generation

In [ ]:
def generate_caption(frame):
    """Generate caption from frame"""
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(rgb_frame)
    
    inputs = processor(pil_image, return_tensors="pt").to(device)
    
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=30)
    
    return processor.decode(output_ids[0], skip_special_tokens=True)

print("Caption generator ready")

## Enhanced Monitoring System

In [ ]:
def run_enhanced_monitor(update_interval=2.0, enable_alerts=True):
    """
    Enhanced real-time monitoring with alert system
    
    Args:
        update_interval: Seconds between caption updates
        enable_alerts: Enable sound/notification alerts
    
    Controls:
        SPACE: Force update
        S: Manual screenshot
        A: Toggle alerts on/off
        Q: Quit
    """
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("Error: Cannot access webcam")
        return
    
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    
    # State
    current_caption = "Initializing..."
    classification = "safe"
    hazard_type = "none"
    severity = "none"
    alert_color = (0, 255, 0)
    last_update = time.time()
    frame_count = 0
    fps = 0
    alerts_enabled = enable_alerts
    
    print("\n" + "="*80)
    print("ENHANCED SAFETY MONITOR WITH ALERTS")
    print("="*80)
    print(f"Alerts: {'ENABLED' if alerts_enabled else 'DISABLED'}")
    print(f"Update interval: {update_interval}s")
    print(f"Alert cooldown: {alert_system.alert_cooldown}s")
    print("\nControls:")
    print("  SPACE - Force immediate update")
    print("  S     - Take screenshot")
    print("  A     - Toggle alerts on/off")
    print("  Q     - Quit")
    print("="*80 + "\n")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # FPS calculation
        frame_count += 1
        if frame_count % 10 == 0:
            current_time = time.time()
            fps = 10 / (current_time - last_update + 0.001)
        
        current_time = time.time()
        time_since_update = current_time - last_update
        
        key = cv2.waitKey(1) & 0xFF
        
        # Toggle alerts
        if key == ord('a'):
            alerts_enabled = not alerts_enabled
            print(f"\n🔔 Alerts {'ENABLED' if alerts_enabled else 'DISABLED'}")
        
        # Update caption
        if time_since_update >= update_interval or key == ord(' '):
            print(f"\nAnalyzing frame...")
            
            start_time = time.time()
            current_caption = generate_caption(frame)
            classification, hazard_type, severity, alert_color = classifier.classify(current_caption)
            inference_time = time.time() - start_time
            
            print(f"Caption: {current_caption}")
            print(f"Classification: {classification} | Type: {hazard_type} | Severity: {severity}")
            
            # Trigger alert if hazard detected
            if classification == 'hazard' and severity in ['high', 'medium'] and alerts_enabled:
                alert_system.trigger_alert(current_caption, hazard_type, severity, frame)
            
            last_update = current_time
        
        # Create display
        display_frame = frame.copy()
        h, w = display_frame.shape[:2]
        
        # Alert border
        if classification == 'hazard' and severity in ['high', 'medium']:
            thickness = 15 if severity == 'high' else 8
            cv2.rectangle(display_frame, (0, 0), (w, h), alert_color, thickness)
        
        # Info overlay
        overlay_height = 150
        overlay = np.zeros((overlay_height, w, 3), dtype=np.uint8)
        
        # Status
        status_text = "🚨 HAZARD DETECTED" if classification == 'hazard' else "✓ SAFE"
        status_color = alert_color if classification == 'hazard' else (0, 255, 0)
        cv2.putText(overlay, status_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, status_color, 2)
        
        # Caption
        caption_display = current_caption[:55] + "..." if len(current_caption) > 55 else current_caption
        cv2.putText(overlay, f"Scene: {caption_display}", (10, 65),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        # Details
        cv2.putText(overlay, f"Type: {hazard_type} | Severity: {severity.upper()}", (10, 95),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
        
        # Stats
        alert_status = "ON" if alerts_enabled else "OFF"
        cv2.putText(overlay, 
                   f"FPS: {fps:.1f} | Alerts: {alert_status} | Total: {alert_system.total_alerts}", 
                   (10, 125), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (150, 150, 150), 1)
        
        # Combine
        display_frame = np.vstack([display_frame, overlay])
        
        cv2.imshow('Enhanced Safety Monitor with Alerts', display_frame)
        
        # Handle keys
        if key == ord('q'):
            print("\n" + "="*80)
            print("MONITORING STOPPED")
            print("="*80)
            break
        elif key == ord('s'):
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"manual_screenshot_{timestamp}.jpg"
            cv2.imwrite(filename, display_frame)
            print(f"📸 Screenshot saved: {filename}")
    
    # Cleanup and save
    cap.release()
    cv2.destroyAllWindows()
    
    # Save alert log
    log_file = alert_system.save_log()
    
    # Print summary
    stats = alert_system.get_stats()
    print("\n" + "="*80)
    print("SESSION SUMMARY")
    print("="*80)
    print(f"Total alerts triggered: {stats['total_alerts']}")
    print(f"Alerts logged: {stats['alert_count']}")
    if stats['hazard_types']:
        print("\nHazards detected:")
        for hazard, count in stats['hazard_types'].items():
            print(f"  {hazard}: {count}")
    if log_file:
        print(f"\nAlert log saved: {log_file}")
    print("="*80)

print("Enhanced monitor ready")

In [ ]:
import subprocess

# Send a notification to register Python with macOS
script = 'display notification "Python notification test - check top right!" with title "Python Alert"'
result = subprocess.run(['osascript', '-e', script], capture_output=True, text=True)

print("Notification sent!")
print("Check the top-right corner of your screen NOW")

## Start Enhanced Monitoring

**Run this to start the system with alerts enabled**

In [ ]:
# Start monitoring with alerts
run_enhanced_monitor(update_interval=2.0, enable_alerts=True)

## View Alert History

In [ ]:
# View most recent alert log
import glob

log_files = sorted(glob.glob('logs/alert_log_*.json'))

if log_files:
    latest_log = log_files[-1]
    print(f"Latest alert log: {latest_log}\n")
    
    with open(latest_log, 'r') as f:
        alerts = json.load(f)
    
    print(f"Total alerts: {len(alerts)}\n")
    print("Recent alerts:")
    print("="*100)
    
    for alert in alerts[-10:]:
        print(f"\n{alert['timestamp']}")
        print(f"  Type: {alert['hazard_type']} | Severity: {alert['severity']}")
        print(f"  Caption: {alert['caption']}")
else:
    print("No alert logs found. Run the monitor first!")

In [ ]:
import subprocess
subprocess.run(['osascript', '-e', 'display notification "Testing notifications!" with title "Safety Monitor Test"'])
print("Check top-right corner of screen NOW!")



---

## Features Summary

**Alert System:**
- 🔊 Sound alerts (system beep)
- 📱 Desktop notifications (Mac)
- 📸 Auto-screenshot on hazard
- 📝 JSON log of all alerts
- ⏱️ Configurable cooldown (default 5s)

**Controls:**
- SPACE: Force immediate update
- S: Manual screenshot
- A: Toggle alerts on/off
- Q: Quit and save logs

**Output Files:**
- `alerts/` - Auto-saved hazard screenshots
- `logs/` - Alert logs in JSON format